# 이진 분류 실험 ① — 기본 피처만으로 선행 연구 방식 재현
1. 피처는 논문의 "기본 피처"만 (Multi-GNN `data_loading.py` 와 동일): 엣지 = Timestamp, Amount Received, Received Currency, Payment Format / 노드 = 전부 1
2. 불균형 처리는 가중 손실만 (Multi-GNN `w_ce = [1, 6.27]`, 트리는 `scale_pos_weight = 6`). 다운샘플링 없음
3. GNN 은 FraudGT·Multi-GNN 처럼 `LinkNeighborLoader` 로 target 2048개가 서브그래프 하나를 공유. 이웃 샘플링 2홉
4. 에폭은 FraudGT 처럼 "배치 N개" 로 고정 (전체 1회 통과가 아님)
5. 검증·테스트 표본과 지표는 05 샘플링 실험과 같은 것(`sampling_lib.evaluate`)을 써서 기존 run.json 과 비교 가능
6. 이 노트북이 기준선. 다음 노트북부터 피처를 하나씩 추가하며 성능 변화를 본다 (GFP Table 5, BlazingAML Table 2 방식)

# 선행 연구 설정 요약
| | Multi-GNN (Egressy 2024) | FraudGT (Lin 2024) | 이 노트북 |
|---|---|---|---|
| 엣지 피처 | Timestamp, Amount Received, Received Currency, Payment Format (z-norm) | 같음 | 같음 |
| 노드 피처 | 전부 1 (+ ego ID) | 같음 | 같음 |
| 역방향 MP | 있음 (`rev_to` 간선 타입) | 있음 | 있음 (역방향 간선 + 방향 플래그 1열) |
| 이웃 샘플링 | 2홉 × 100 | 2홉 × 50 | `NUM_NEIGHBORS` 로 설정 |
| 배치 | 8192 target | 2048 target | `BATCH_SIZE` |
| 손실 | CE, 가중치 [1, 6.27] | CE, 가중치 [1, 6] | CE, 가중치 `CLASS_WEIGHT` |
| 에폭 | 전체 1회 통과 | 배치 256개 | 배치 `ITER_PER_EPOCH` 개 |
| 에폭 선택 | 검증 F1 (argmax) | 검증 F1 | 검증 AP (05 실험과 통일) + F1@0.5 도 기록 |
| 분할 | 시간순 60/20/20 | 같음 | 주 단위 8/8~10/2 · ~10/16 · ~10/30 (05 실험과 동일) |

역방향 MP 를 `to`/`rev_to` 두 간선 타입(hetero) 대신 "역방향 간선을 같은 그래프에 넣고 방향 플래그 피처를 1열 붙이는" 동종 그래프로 구현했다. 메시지가 양방향으로 흐른다는 점은 같고 코드가 단순하다.

# 공통 부분 (작업 준비, 데이터 준비)

## 0. 작업 준비

In [1]:
# !pip install --break-system-packages --no-cache-dir -q lightgbm xgboost
# !pip install --break-system-packages --no-cache-dir -q duckdb pyarrow

In [2]:
import sys, json, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm.auto import tqdm

In [3]:
# 05 샘플링 실험의 공통 모듈 (경로, 분할, 검증 표본, 지표) 를 그대로 쓴다
sys.path.append(str(Path.cwd().parent))
import sampling_lib as L

In [4]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DEVICE

device(type='cuda')

In [5]:
# ---- 실험 설정 ------------------------------------------------------------
DATASET = 'HI-Large'          # 처음엔 'HI-Small' 로 한 번 끝까지 돌려서 코드가 도는지 확인하고 Large 로 간다
SMOKE = False                 # True 면 학습 target·에폭을 줄여서 빨리 한 바퀴만 돈다

# 분할 (05 실험과 동일: 월~일 주 단위, 11월 미사용)
TRAIN_WINDOW = L.TRAIN        # '2022-08-08:2022-10-02'
VAL_WINDOW   = L.VAL          # '2022-10-03:2022-10-16'
TEST_WINDOW  = L.TEST         # '2022-10-17:2022-10-30'
VAL_NEG_RATE = 0.02           # 검증·테스트 표본: 양성 전부 + 정상 2% (가중치 1/rate). 05 와 같은 파일을 공유

# 기본 피처 (Multi-GNN data_loading.py 의 edge_features 그대로)
BASIC_EDGE_FEATURES = ['timestamp', 'amount_received', 'received_currency', 'payment_format']

# 트리 모델
SCALE_POS_WEIGHT = 6          # Altman 2023 / GFP 탐색 범위 1~10 의 중간
TREE_ROUNDS = 500
TREE_EARLY_STOP = 20

# GNN (Multi-GNN model_settings.json 'gin' 기준)
NUM_NEIGHBORS = [50, 50]      # FraudGT 50 / Multi-GNN·v10 100. 메모리 보고 올린다
BATCH_SIZE = 2048             # target 간선 수 (FraudGT)
ITER_PER_EPOCH = 256          # 에폭 = 배치 256개 = 52만 target (FraudGT). 전체 1회 통과가 아님
EPOCHS = 30
PATIENCE = 5
CLASS_WEIGHT = [1.0, 6.27]    # Multi-GNN w_ce1, w_ce2
N_HIDDEN = 66
N_GNN_LAYERS = 2
LR = 0.006
DROPOUT = 0.01
FINAL_DROPOUT = 0.1
NUM_WORKERS = 0               # 샘플링 워커. 메모리가 빠듯하면 0

if SMOKE:
    ITER_PER_EPOCH, EPOCHS, TREE_ROUNDS = 8, 2, 20

OUT_DIR = Path.cwd() / 'results' / DATASET / 'basic_features'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR

PosixPath('/workspace/money_laundry_ai_/code/binary_classification/results/HI-Large/basic_features')

In [6]:
# sampling_lib 은 DATASET 상수를 모듈 로드 시점에 읽으므로, Small 로 돌릴 땐 경로를 바꿔 준다
L.DATASET = DATASET
L.BASE = L.ROOT / 'parquets' / DATASET
L.SAMPLING = L.BASE / 'sampling'
TX_PATH = L.BASE / 'transactions.parquet'
assert TX_PATH.is_file(), f'01_prepare_parquet 을 {DATASET} 으로 먼저 실행: {TX_PATH}'
TX_PATH

PosixPath('/workspace/money_laundry_ai_/parquets/HI-Large/transactions.parquet')

## 1. 데이터 준비 — 거래 전체에서 기본 피처 4개만

- `transactions.parquet` 에서 라벨과 기본 피처에 필요한 열만 읽는다 (중복 거래는 01 에서 대표 1건만 남긴 상태)
- 통화·결제 방식은 Multi-GNN 처럼 정수 코드로 (논문은 one-hot 이 아니라 코드값을 z-norm 한다. 그대로 따른다)
- 시각은 "데이터 시작부터 초" 로 (Multi-GNN `Timestamp - min`)
- z-norm 통계는 **학습 구간 거래**로만 구해서 검증·테스트에도 같은 값을 쓴다

In [7]:
con = L.connect(threads=8, memory='12GB')
train_a, train_b = L.parse_window(TRAIN_WINDOW)

# 통화·결제 방식 → 정수 코드. 학습 구간에서 본 값에 0,1,2,... 를 주고, 못 본 값은 -1 (문자열을 pandas 에 올리지 않는다)
currency_codes = con.execute(f"""
    SELECT receiving_currency AS v, row_number() OVER (ORDER BY receiving_currency) - 1 AS code
    FROM (SELECT DISTINCT receiving_currency FROM read_parquet({L.q(TX_PATH)})
          WHERE day BETWEEN DATE {L.q(train_a)} AND DATE {L.q(train_b)})""").df()
format_codes = con.execute(f"""
    SELECT payment_format AS v, row_number() OVER (ORDER BY payment_format) - 1 AS code
    FROM (SELECT DISTINCT payment_format FROM read_parquet({L.q(TX_PATH)})
          WHERE day BETWEEN DATE {L.q(train_a)} AND DATE {L.q(train_b)})""").df()
con.register('currency_codes', currency_codes); con.register('format_codes', format_codes)

tx = con.execute(f"""
    SELECT t.edge_id, t.src::INT AS src, t.dst::INT AS dst, CAST(t.day AS TIMESTAMP) AS day,
           epoch(t.ts) - (SELECT epoch(min(ts)) FROM read_parquet({L.q(TX_PATH)}))  AS timestamp,   -- 데이터 시작부터 초
           t.received::FLOAT                        AS amount_received,
           coalesce(c.code, -1)::SMALLINT           AS received_currency,
           coalesce(f.code, -1)::SMALLINT           AS payment_format,
           t.is_laundering::TINYINT                 AS y
    FROM read_parquet({L.q(TX_PATH)}) t
    LEFT JOIN currency_codes c ON c.v = t.receiving_currency
    LEFT JOIN format_codes   f ON f.v = t.payment_format
    ORDER BY t.edge_id""").df()

tx.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(179702079, 9)

In [8]:
tx.head()

,edge_id,src,dst,day,timestamp,amount_received,received_currency,payment_format,y
0,1,3,3,2022-08-01,0.0,19.180000,12,5,0
1,2,25,78803,2022-08-01,0.0,17955.109375,12,3,0
2,3,31,86028,2022-08-01,0.0,1199.829956,12,2,0
3,4,39,269980,2022-08-01,0.0,148565.453125,12,6,0
4,5,50,29866,2022-08-01,0.0,227247.421875,12,3,0


In [9]:
currency_codes, format_codes

(                    v  code
 0   Australian Dollar     0
 1             Bitcoin     1
 2         Brazil Real     2
 3     Canadian Dollar     3
 4                Euro     4
 5        Mexican Peso     5
 6               Ruble     6
 7               Rupee     7
 8         Saudi Riyal     8
 9              Shekel     9
 10        Swiss Franc    10
 11           UK Pound    11
 12          US Dollar    12
 13                Yen    13
 14               Yuan    14,
               v  code
 0           ACH     0
 1       Bitcoin     1
 2          Cash     2
 3        Cheque     3
 4   Credit Card     4
 5  Reinvestment     5
 6          Wire     6)

In [10]:
def window_mask(df, window):
    a, b = L.parse_window(window)
    return ((df['day'] >= pd.Timestamp(a)) & (df['day'] <= pd.Timestamp(b))).to_numpy()

in_train = window_mask(tx, TRAIN_WINDOW)

# z-norm: 학습 구간 통계로 전 구간 정규화 (Multi-GNN z_norm 과 같은 꼴)
basic = tx[BASIC_EDGE_FEATURES].to_numpy(np.float32)
mu = basic[in_train].mean(0)
sd = basic[in_train].std(0) + 1e-6
basic_z = (basic - mu) / sd
del basic

pd.DataFrame({'feature': BASIC_EDGE_FEATURES, 'mean(train)': mu, 'std(train)': sd})

,feature,mean(train),std(train)
0,timestamp,1.375241e+06,1.698260e+06
1,amount_received,5.482256e+06,1.804425e+09
2,received_currency,2.623063e+00,5.673916e+00
3,payment_format,1.311532e+00,1.498233e+00


In [11]:
# 라벨 분포 확인 (구간별)
for name, w in [('train', TRAIN_WINDOW), ('val', VAL_WINDOW), ('test', TEST_WINDOW)]:
    m = window_mask(tx, w)
    print(f'{name:5s} {w}  거래 {m.sum():>12,}  세탁 {tx.loc[m, "y"].sum():>8,}  비율 {tx.loc[m, "y"].mean()*100:.3f}%')

train 2022-08-08:2022-10-02  거래  102,336,629  세탁  110,671  비율 0.108%
val   2022-10-03:2022-10-16  거래   24,413,046  세탁   33,795  비율 0.138%
test  2022-10-17:2022-10-30  거래   26,808,230  세탁   34,460  비율 0.129%


## 2. 검증·테스트 표본 (05 실험과 공유)

검증 구간 전체(수천만 건)를 매 에폭 평가할 수 없으므로 **양성 전부 + 정상 2% (가중치 1/0.02)** 표본으로 평가한다.
가중 지표는 전체 평가의 불편추정이고, 05 실험의 모든 조건이 같은 파일을 쓴다 → 결과를 그대로 비교할 수 있다.

In [12]:
def val_sample(con, window, rate, seed=0):
    """고정 검증 표본 (05_sampling_train.val_sample 과 같은 규칙·같은 파일). 없으면 만든다."""
    a, b = L.parse_window(window)
    path = L.SAMPLING / 'val' / f'{a}_{b}_neg{rate:g}.parquet'
    if path.exists():
        return pd.read_parquet(path)
    lab = L.load_labels(con, (a, b))
    rng = np.random.default_rng(seed)
    parts = [lab[lab.y == 1].assign(w=1.0)]
    for day, g in lab[lab.y == 0].groupby('day'):
        k = max(1, int(round(len(g) * rate)))
        parts.append(g.iloc[rng.choice(len(g), k, replace=False)].assign(w=len(g) / k))
    v = pd.concat(parts).sort_values('edge_id').reset_index(drop=True)
    path.parent.mkdir(parents=True, exist_ok=True)
    v.to_parquet(path, index=False)
    return v

val_df  = val_sample(con, VAL_WINDOW,  VAL_NEG_RATE)
test_df = val_sample(con, TEST_WINDOW, VAL_NEG_RATE)
con.close()

val_df.shape, test_df.shape

((521382, 7), (569935, 7))

In [13]:
val_df.head()

,edge_id,day,src,dst,y,in_pattern,w
0,117269768,2022-10-03,11210,1050074,0,False,49.999844
1,117269781,2022-10-03,14672,1345482,0,False,49.999844
2,117269788,2022-10-03,15785,89179,0,False,49.999844
3,117269866,2022-10-03,41306,1339225,0,False,49.999844
4,117269893,2022-10-03,47308,410704,0,False,49.999844


In [14]:
# 표본의 edge_id → tx 행 위치 (tx 는 edge_id 오름차순이라 searchsorted 로 바로 찾는다)
all_edge_ids = tx['edge_id'].to_numpy()

def rows_of(edge_ids):
    pos = np.searchsorted(all_edge_ids, edge_ids)
    assert (all_edge_ids[pos] == edge_ids).all(), 'tx 에 없는 edge_id'
    return pos

val_idx  = rows_of(val_df['edge_id'].to_numpy())
test_idx = rows_of(test_df['edge_id'].to_numpy())

assert (tx['y'].to_numpy()[val_idx] == val_df['y'].to_numpy()).all()
len(val_idx), len(test_idx)

(521382, 569935)

In [15]:
def evaluate_sample(sample_df, score):
    """05 와 같은 가중 지표 + 논문 비교용 F1@0.5 (임계값 0.5 고정, 가중)."""
    y = sample_df['y'].to_numpy(); w = sample_df['w'].to_numpy()
    unmatched = (y == 1) & ~sample_df['in_pattern'].to_numpy()
    m = L.evaluate(y, score, w, unmatched)
    pred = score >= 0.5
    tp = (w * (pred & (y == 1))).sum(); fp = (w * (pred & (y == 0))).sum(); fn = (w * (~pred & (y == 1))).sum()
    m['F1_at_0.5'] = float(2 * tp / max(2 * tp + fp + fn, 1e-12))
    return m

results = {}   # 모델 이름 → {'val': 지표, 'test': 지표, ...}

# 트리 모델 기준선 (LightGBM, XGBoost)

## 3. 기본 피처 4개로 트리 모델

- 입력은 기본 피처 4개뿐 (계좌 ID 없음). GFP 논문 Table 4 의 "LightGBM basic / XGBoost basic" 에 해당 (HI-Large 24.5 / 10.6)
- 불균형은 `scale_pos_weight` 만. 검증 표본의 가중 AP 로 early stopping

In [16]:
train_mask = in_train
X_train = basic_z[train_mask]
y_train = tx['y'].to_numpy()[train_mask]
X_val, y_val, w_val = basic_z[val_idx], val_df['y'].to_numpy(), val_df['w'].to_numpy()
X_test = basic_z[test_idx]

if SMOKE:
    keep = np.random.default_rng(SEED).choice(len(X_train), min(len(X_train), 200_000), replace=False)
    X_train, y_train = X_train[keep], y_train[keep]

X_train.shape, y_train.mean()

((102336629, 4), np.float64(0.0010814407420045075))

In [17]:
import lightgbm as lgb

t0 = time.time()
lgb_model = lgb.LGBMClassifier(
    n_estimators=TREE_ROUNDS, learning_rate=0.05, num_leaves=63,
    scale_pos_weight=SCALE_POS_WEIGHT, subsample=0.8, subsample_freq=1, colsample_bytree=1.0,
    random_state=SEED, n_jobs=8, verbose=-1,
)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)], eval_sample_weight=[w_val], eval_metric='average_precision',
    callbacks=[lgb.early_stopping(TREE_EARLY_STOP, verbose=False)],
)
lgb_sec = time.time() - t0

results['lightgbm_basic'] = dict(
    val=evaluate_sample(val_df, lgb_model.predict_proba(X_val)[:, 1]),
    test=evaluate_sample(test_df, lgb_model.predict_proba(X_test)[:, 1]),
    best_iteration=int(lgb_model.best_iteration_ or TREE_ROUNDS), train_sec=round(lgb_sec),
)
results['lightgbm_basic']['val']

/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


{'AP': 0.0486452419870441,
 'F1': 0.12307611250968688,
 'precision': 0.07589627063761037,
 'recall': 0.3252848054445924,
 'threshold': 0.9892289816634532,
 'recall_at_1pct': 0.43598165409084183,
 'unmatched_recall_at_1pct': 0.06936838262139466,
 'ECE': 0.009342336488901716,
 'F1_at_0.5': 0.11320267377295658}

In [18]:
import xgboost as xgb

t0 = time.time()
xgb_model = xgb.XGBClassifier(
    n_estimators=TREE_ROUNDS, learning_rate=0.05, max_depth=8,
    scale_pos_weight=SCALE_POS_WEIGHT, subsample=0.8, colsample_bytree=1.0,
    tree_method='hist', device='cpu',          # 최신 xgboost GPU 커널이 V100(sm_70) 을 지원하지 않아 CPU 로
    eval_metric='aucpr', early_stopping_rounds=TREE_EARLY_STOP,
    random_state=SEED, n_jobs=8,
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], sample_weight_eval_set=[w_val], verbose=False)
xgb_sec = time.time() - t0

results['xgboost_basic'] = dict(
    val=evaluate_sample(val_df, xgb_model.predict_proba(X_val)[:, 1]),
    test=evaluate_sample(test_df, xgb_model.predict_proba(X_test)[:, 1]),
    best_iteration=int(xgb_model.best_iteration), train_sec=round(xgb_sec),
)
results['xgboost_basic']['val']

{'AP': 0.05614451415704122,
 'F1': 0.13011548473756204,
 'precision': 0.09053025415320519,
 'recall': 0.23121763574493268,
 'threshold': 0.45901623368263245,
 'recall_at_1pct': 0.4434975588104749,
 'unmatched_recall_at_1pct': 0.06381891201168309,
 'ECE': 0.009510450346360003,
 'F1_at_0.5': 0.1272345578050196}

In [19]:
# 피처 중요도 — 기본 피처 4개 중 뭘 보는지
xgb_gain = xgb_model.get_booster().get_score(importance_type='gain')
pd.DataFrame({
    'feature': BASIC_EDGE_FEATURES,
    'lightgbm_gain': lgb_model.booster_.feature_importance('gain'),
    'xgboost_gain': [xgb_gain.get(f'f{i}', 0.0) for i in range(len(BASIC_EDGE_FEATURES))],
})

,feature,lightgbm_gain,xgboost_gain
0,timestamp,3.026504e+06,98.148163
1,amount_received,2.554977e+07,584.453247
2,received_currency,1.793715e+07,515.631104
3,payment_format,1.552910e+07,15143.089844


In [20]:
# 그래프를 만들기 전에 트리 학습용 배열을 지운다 (X_train 1.6GB + 트리 내부 데이터 수 GB). 모델 객체는 가볍게 남는다
import gc
del X_train, y_train
gc.collect()

121

# GNN (Multi-GNN 구조, FraudGT 식 학습)

## 4. 그래프 만들기

- 그래프는 Multi-GNN 규칙: 학습 그래프 = 학습 구간 **끝까지**의 거래 (8/7 이전 포함), 검증 그래프 = 검증 구간 끝까지, 테스트 그래프 = 테스트 구간 끝까지
- 역방향 메시지 패싱: 간선을 뒤집은 복사본을 같은 그래프에 넣고 `is_reverse` 플래그를 엣지 피처 마지막 열에 붙인다
- 노드 피처는 전부 1. ego ID(판정 대상 간선의 양끝 = 1) 는 배치 단위로 붙인다 (Multi-GNN `AddEgoIds`)
- **메모리** (HI-Large 는 양방향 3.4억 간선): 두 번 터진 뒤 정리한 규칙
  - `LinkNeighborLoader` 의 샘플러는 간선을 도착 노드순으로 정렬한 사본(간선당 16바이트)을 그래프마다 만든다 → 그래프를 만들 때
    **미리 도착 노드순으로 정렬**해 두고 `is_sorted=True` 를 주면 사본을 안 만든다
  - 엣지 피처 float16, 라벨 uint8 로 저장하고 배치에서만 float32 / long
  - 학습·검증 그래프만 먼저 만들고, 테스트 그래프는 학습이 끝나 학습 그래프를 지운 뒤 만든다
  - 큰 객체(tx, basic_z, 트리 학습 배열)는 쓰고 나면 바로 지우고 `malloc_trim` 으로 OS 에 돌려준다

In [ ]:
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader
import ctypes, gc, psutil

def trim():
    """지운 메모리를 OS 에 돌려주고(glibc 는 안 돌려줄 때가 많다) 현재 RSS 를 찍는다"""
    gc.collect(); ctypes.CDLL('libc.so.6').malloc_trim(0)
    print(f'RSS {psutil.Process().memory_info().rss / 1e9:.1f} GB')

NUM_NODES = int(max(tx['src'].max(), tx['dst'].max())) + 1

# 구간 끝 = tx 행 번호 (tx 는 시간순이라 "끝 날짜까지" = 앞부분)
def rows_until(window):
    _, end = L.parse_window(window)
    return int((tx['day'] <= pd.Timestamp(end)).sum())

N_TRAIN, N_VAL, N_TEST = rows_until(TRAIN_WINDOW), rows_until(VAL_WINDOW), rows_until(TEST_WINDOW)
print(f'간선 수 (정방향): 학습까지 {N_TRAIN:,}  검증까지 {N_VAL:,}  테스트까지 {N_TEST:,}')

# 그래프 재료만 남기고 tx, basic_z 는 지운다 (각각 7GB, 2.9GB)
SRC = tx['src'].to_numpy().copy(); DST = tx['dst'].to_numpy().copy(); Y = tx['y'].to_numpy().astype(np.uint8)
BASIC16 = torch.as_tensor(basic_z).to(torch.float16)        # [N, 4]
del tx, basic_z
trim()

In [ ]:
def build_graph(n_rows):
    """tx 앞 n_rows 건으로 양방향 그래프. 간선을 도착 노드(col)순으로 정렬해 저장한다 (샘플러 is_sorted=True 용).
    반환: (Data, fwd_pos) — fwd_pos[j] = tx 행 j 의 정방향 간선이 정렬 후 몇 번째인지 (target 지정용)
    임시 복사본을 안 만들려고 텐서를 미리 할당하고 제자리에서 채운다 (HI-Large 검증 그래프 기준 순간 최대 약 12GB)"""
    t0 = time.time(); m = n_rows; E = 2 * m
    src = torch.as_tensor(SRC[:m]); dst = torch.as_tensor(DST[:m])
    ei = torch.empty(2, E, dtype=torch.long)
    ei[1, :m] = dst; ei[1, m:] = src                        # col = 도착 노드 (정방향 dst, 역방향 src)
    perm = torch.argsort(ei[1])                              # 정렬 키 = 도착 노드
    ei[1] = ei[1][perm]
    ei[0, :m] = src; ei[0, m:] = dst; ei[0] = ei[0][perm]    # row = 출발 노드
    del src, dst
    rev = perm >= m                                           # 정렬 후 e 번째 간선이 역방향인가
    perm.remainder_(m); orig = perm                           # 정렬 후 e 번째 간선 ← tx 행 orig[e]  (perm 을 제자리에서 재사용)
    attr = torch.empty(E, BASIC16.shape[1] + 1, dtype=torch.float16)
    attr[:, :-1] = BASIC16[orig]; attr[:, -1] = rev.to(torch.float16)
    fwd = ~rev; del rev
    y = torch.zeros(E, dtype=torch.uint8); y[fwd] = torch.as_tensor(Y)[orig[fwd]]
    fwd_pos = torch.empty(m, dtype=torch.long); fwd_pos[orig[fwd]] = torch.nonzero(fwd).squeeze(1)
    del orig, perm, fwd
    data = Data(x=torch.ones(NUM_NODES, 1), edge_index=ei, edge_attr=attr, y=y, num_nodes=NUM_NODES)
    print(f'그래프 {E:,} 간선 (정렬됨)  {time.time()-t0:.0f}s'); return data, fwd_pos

def targets_of(fwd_pos, tx_rows, graph):
    t = fwd_pos[torch.as_tensor(np.asarray(tx_rows))]
    assert (graph.y[t].numpy() == Y[np.asarray(tx_rows)]).all(), 'target 위치 오류'
    return t

train_graph, fwd_pos = build_graph(N_TRAIN)
train_targets = targets_of(fwd_pos, np.flatnonzero(train_mask), train_graph); del fwd_pos
if SMOKE:
    train_targets = train_targets[torch.randperm(len(train_targets))[:50_000]]
trim()

In [ ]:
val_graph, fwd_pos = build_graph(N_VAL)
val_targets = targets_of(fwd_pos, val_idx, val_graph); del fwd_pos
trim()
len(train_targets), len(val_targets)

In [ ]:
class AddEgoIdsAndTargetAttr:
    """배치 변환 3가지
    1) ego ID: 판정 대상(seed) 간선의 양끝 노드에 1, 나머지 0 인 피처 열을 x 에 붙인다 (Multi-GNN AddEgoIds)
    2) seed 간선의 원 엣지 피처를 batch.target_attr 로 싣는다 — 이웃 샘플링 결과에 seed 간선 자체가 빠질 수 있어서
       그래프 edge_attr 에서 직접 꺼낸다 (batch.input_id = targets 안에서의 위치)
    3) 저장용 float16 / uint8 → 계산용 float32 / long"""
    def __init__(self, graph, targets):
        self.edge_attr, self.targets = graph.edge_attr, targets
    def __call__(self, batch):
        ego = torch.zeros(batch.num_nodes, 1)
        ego[batch.edge_label_index.flatten()] = 1
        batch.x = torch.cat([batch.x, ego], 1)
        batch.edge_attr = batch.edge_attr.float()
        batch.target_attr = self.edge_attr[self.targets[batch.input_id]].float()   # [B, 5]
        batch.edge_label = batch.edge_label.long()
        return batch

def make_loader(graph, targets, shuffle):
    return LinkNeighborLoader(
        graph, num_neighbors=NUM_NEIGHBORS,
        edge_label_index=graph.edge_index[:, targets], edge_label=graph.y[targets],
        batch_size=BATCH_SIZE, shuffle=shuffle, transform=AddEgoIdsAndTargetAttr(graph, targets),
        is_sorted=True,                       # 그래프를 도착 노드순으로 정렬해 뒀으므로 샘플러가 정렬 사본을 안 만든다
        num_workers=NUM_WORKERS, persistent_workers=NUM_WORKERS > 0,
    )

train_loader = make_loader(train_graph, train_targets, shuffle=True)
val_loader   = make_loader(val_graph,   val_targets,   shuffle=False)
trim()

In [ ]:
# 배치 하나 꺼내서 모양 확인 — 2048 target 이 서브그래프 하나를 공유한다
batch = next(iter(train_loader))
batch

In [ ]:
print('노드', batch.num_nodes, ' 간선', batch.num_edges, ' target', batch.edge_label_index.shape[1],
      ' 양성', int(batch.edge_label.sum()), ' x 열', batch.x.shape[1], '(1 + ego)', ' target_attr', tuple(batch.target_attr.shape))

## 5. GNN 모델 — Multi-GNN `GINe` 그대로

- `GINe` (Egressy 2024 `models.py`): 노드·엣지 임베딩 → GINEConv × 2 (+ BatchNorm, residual) → [src, dst, edge] 연결 → MLP → 2클래스
- `edge_updates=True` 가 논문의 **GIN+EU** (층마다 엣지 임베딩도 갱신). 둘 다 돌린다
- 엣지 피처 차원 = 기본 4 + is_reverse 1 = 5. 노드 = 1 + ego 1 = 2

In [ ]:
from torch_geometric.nn import GINEConv, BatchNorm

class GINe(nn.Module):
    def __init__(self, num_features, edge_dim, n_hidden=66, num_gnn_layers=2, n_classes=2,
                 edge_updates=False, dropout=0.0, final_dropout=0.5):
        super().__init__()
        self.n_hidden, self.num_gnn_layers, self.edge_updates = n_hidden, num_gnn_layers, edge_updates
        self.node_emb = nn.Linear(num_features, n_hidden)
        self.edge_emb = nn.Linear(edge_dim, n_hidden)
        self.convs, self.emlps, self.batch_norms = nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        for _ in range(num_gnn_layers):
            self.convs.append(GINEConv(nn.Sequential(nn.Linear(n_hidden, n_hidden), nn.ReLU(), nn.Linear(n_hidden, n_hidden)), edge_dim=n_hidden))
            if edge_updates:
                self.emlps.append(nn.Sequential(nn.Linear(3 * n_hidden, n_hidden), nn.ReLU(), nn.Linear(n_hidden, n_hidden)))
            self.batch_norms.append(BatchNorm(n_hidden))
        self.dropout = dropout
        self.mlp = nn.Sequential(nn.Linear(n_hidden * 3, 50), nn.ReLU(), nn.Dropout(final_dropout),
                                 nn.Linear(50, 25), nn.ReLU(), nn.Dropout(final_dropout),
                                 nn.Linear(25, n_classes))

    def forward(self, x, edge_index, edge_attr, target_index, target_attr):
        """target_index: 판정할 간선의 (src, dst) 노드 번호 [2, B], target_attr: 그 간선의 원 피처 [B, edge_dim]. 반환 [B, 2] 로짓"""
        src, dst = edge_index
        x = self.node_emb(x)
        edge_attr = self.edge_emb(edge_attr)
        target_emb = self.edge_emb(target_attr)          # seed 간선 임베딩 (edge_updates 는 이웃 간선에만 적용)
        for i in range(self.num_gnn_layers):
            x = (x + F.relu(self.batch_norms[i](self.convs[i](x, edge_index, edge_attr)))) / 2
            x = F.dropout(x, self.dropout, self.training)
            if self.edge_updates:
                edge_attr = edge_attr + self.emlps[i](torch.cat([x[src], x[dst], edge_attr], -1)) / 2
        # target 간선의 [src 임베딩, dst 임베딩, 엣지 임베딩]
        h = torch.cat([x[target_index[0]], x[target_index[1]]], 1).relu()
        return self.mlp(torch.cat([h, target_emb], 1))

In [ ]:
def forward_batch(model, batch):
    batch = batch.to(DEVICE)
    logits = model(batch.x, batch.edge_index, batch.edge_attr, batch.edge_label_index, batch.target_attr)
    return logits, batch.edge_label

# 모양 확인
model = GINe(num_features=2, edge_dim=train_graph.edge_attr.shape[1], n_hidden=N_HIDDEN, num_gnn_layers=N_GNN_LAYERS,
             dropout=DROPOUT, final_dropout=FINAL_DROPOUT).to(DEVICE)
logits, labels = forward_batch(model, batch)
logits.shape, labels.shape

## 6. 학습 — 가중 CE + FraudGT 식 에폭

- 손실: `CrossEntropyLoss(weight=[1, 6.27])`. 배치 안 양성은 자연 비율(약 0.1%)로 들어온다 → 양성 손실 비중 ≈ 0.7%
- 에폭 = `ITER_PER_EPOCH` 배치. 셔플된 iterator 를 에폭 경계에서 끊지 않고 이어 쓴다 (FraudGT `LoaderWrapper`)
- 에폭마다 검증 표본 평가, 검증 AP 최고 에폭을 저장. 논문 비교용으로 F1@0.5 도 기록

In [ ]:
class InfiniteLoader:
    """셔플된 loader 를 끝까지 다 읽으면 다시 섞어서 이어 준다 (FraudGT LoaderWrapper 의 train 동작)."""
    def __init__(self, loader):
        self.loader, self.it = loader, iter(loader)
    def next(self):
        try:
            return next(self.it)
        except StopIteration:
            self.it = iter(self.loader)
            return next(self.it)

@torch.no_grad()
def predict_loader(model, loader):
    """loader 순서대로 양성 확률. (shuffle=False 라 targets 순서와 같다)"""
    model.eval(); out = []
    for b in loader:
        logits, _ = forward_batch(model, b)
        out.append(F.softmax(logits, -1)[:, 1].cpu().numpy())
    return np.concatenate(out)

In [ ]:
def train_gnn(name, edge_updates):
    torch.manual_seed(SEED)
    model = GINe(num_features=2, edge_dim=train_graph.edge_attr.shape[1], n_hidden=N_HIDDEN, num_gnn_layers=N_GNN_LAYERS,
                 edge_updates=edge_updates, dropout=DROPOUT, final_dropout=FINAL_DROPOUT).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(CLASS_WEIGHT, device=DEVICE))
    stream = InfiniteLoader(train_loader)
    history, best_ap, stale = [], -np.inf, 0
    ckpt = OUT_DIR / f'{name}.pt'
    t_start = time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train(); losses = []; te = time.time()
        for _ in tqdm(range(ITER_PER_EPOCH), desc=f'{name} epoch {epoch}', leave=False):
            logits, labels = forward_batch(model, stream.next())
            loss = loss_fn(logits, labels)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            losses.append(loss.item())
        score = predict_loader(model, val_loader)
        m = evaluate_sample(val_df, score)
        history.append(dict(epoch=epoch, train_loss=float(np.mean(losses)), epoch_sec=round(time.time() - te), **m))
        print(f"epoch {epoch:2d}  loss {np.mean(losses):.4f}  AP {m['AP']:.4f}  F1 {m['F1']:.4f}  F1@0.5 {m['F1_at_0.5']:.4f}  "
              f"recall@1% {m['recall_at_1pct']:.4f}  ({time.time() - te:.0f}s)")
        if m['AP'] > best_ap:
            best_ap, stale = m['AP'], 0
            torch.save(model.state_dict(), ckpt)
        else:
            stale += 1
            if stale >= PATIENCE:
                break
    # 최고 에폭 모델로 검증·테스트
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
    hist = pd.DataFrame(history); hist.to_csv(OUT_DIR / f'{name}_history.csv', index=False)
    best = hist.loc[hist['AP'].idxmax()]
    results[name] = dict(
        val=evaluate_sample(val_df, predict_loader(model, val_loader)),
        test=None,                                   # 테스트는 7절에서 학습 그래프를 지운 뒤 한 번에
        best_epoch=int(best['epoch']), epochs_run=len(hist), train_sec=round(time.time() - t_start),
        targets_seen=int(best['epoch']) * ITER_PER_EPOCH * BATCH_SIZE,
        gpu_peak_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2) if DEVICE.type == 'cuda' else None,
    )
    return model, hist

In [ ]:
gine_model, gine_hist = train_gnn('gine_basic', edge_updates=False)
results['gine_basic']['val']

In [ ]:
gine_eu_model, gine_eu_hist = train_gnn('gine_eu_basic', edge_updates=True)
results['gine_eu_basic']['val']

In [ ]:
# 학습 곡선 — 에폭 = 52만 target 이므로 x 축은 "본 target 수" 로도 읽을 수 있다
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for name, h in [('GINe', gine_hist), ('GINe+EU', gine_eu_hist)]:
    ax[0].plot(h['epoch'], h['AP'], marker='o', label=name)
    ax[1].plot(h['epoch'], h['train_loss'], marker='o', label=name)
ax[0].set_title('검증 AP'); ax[1].set_title('학습 손실'); ax[0].set_xlabel('epoch'); ax[1].set_xlabel('epoch')
ax[0].legend(); plt.tight_layout(); plt.savefig(OUT_DIR / 'curves.png', dpi=120); plt.show()

# 결과 비교

## 7. 테스트 평가 — 학습 그래프를 지우고 테스트 그래프를 만든다

In [ ]:
# 학습·검증 그래프·loader 를 지우고 테스트 그래프를 만든다 (셋을 동시에 들고 있으면 메모리 초과)
del train_loader, val_loader, train_graph, val_graph, train_targets, val_targets
trim()
test_graph, fwd_pos = build_graph(N_TEST)
test_targets = targets_of(fwd_pos, test_idx, test_graph); del fwd_pos
test_loader = make_loader(test_graph, test_targets, shuffle=False)
trim()

In [ ]:
# 저장해 둔 최고 에폭 체크포인트로 두 모델 테스트
for name, edge_updates in [('gine_basic', False), ('gine_eu_basic', True)]:
    model = GINe(num_features=2, edge_dim=test_graph.edge_attr.shape[1], n_hidden=N_HIDDEN, num_gnn_layers=N_GNN_LAYERS,
                 edge_updates=edge_updates, dropout=DROPOUT, final_dropout=FINAL_DROPOUT).to(DEVICE)
    model.load_state_dict(torch.load(OUT_DIR / f'{name}.pt', map_location=DEVICE, weights_only=True))
    results[name]['test'] = evaluate_sample(test_df, predict_loader(model, test_loader))
    print(name, {k: round(v, 4) for k, v in results[name]['test'].items()})

## 8. 모델별 비교표 + 저장

In [ ]:
METRIC_COLS = ['AP', 'F1', 'F1_at_0.5', 'precision', 'recall', 'recall_at_1pct', 'unmatched_recall_at_1pct', 'ECE']
rows = []
for name, r in results.items():
    for split in ('val', 'test'):
        rows.append(dict(model=name, split=split, **{k: r[split][k] for k in METRIC_COLS}, train_sec=r['train_sec']))
compare_df = pd.DataFrame(rows).set_index(['model', 'split']).round(4)
compare_df

In [ ]:
# 선행 연구 수치와 나란히 (HI-Large, 소수 클래스 F1 %). 논문은 argmax(=0.5) F1 이므로 F1_at_0.5 와 비교한다
prior = pd.DataFrame({
    'model': ['LightGBM basic', 'XGBoost basic', 'GIN', 'GIN+EU', 'GIN+RMP (FraudGT 재실행)'],
    'F1(%)': [24.5, 10.6, 36.3, None, 59.3],
    'source': ['GFP Table 4', 'GFP Table 4', 'FraudGT', '-', 'FraudGT'],
})
prior

In [ ]:
# 05 샘플링 실험(v10 구조, 36피처, 지도 샘플링 r=10) 의 결과와도 비교 — 같은 검증 표본·같은 지표
runs = sorted((L.SAMPLING / 'runs').glob('*/run.json'))
if runs:
    ref = pd.DataFrame([{**{'run': p.parent.name}, **json.loads(p.read_text(encoding='utf-8'))['val']} for p in runs])
    display(ref[['run', 'AP', 'F1', 'recall_at_1pct', 'unmatched_recall_at_1pct']].round(4))

In [ ]:
(OUT_DIR / 'results.json').write_text(json.dumps(dict(
    dataset=DATASET, smoke=SMOKE, features=BASIC_EDGE_FEATURES, windows=dict(train=TRAIN_WINDOW, val=VAL_WINDOW, test=TEST_WINDOW),
    val_neg_rate=VAL_NEG_RATE, scale_pos_weight=SCALE_POS_WEIGHT, class_weight=CLASS_WEIGHT,
    num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE, iter_per_epoch=ITER_PER_EPOCH,
    results=results,
), ensure_ascii=False, indent=2), encoding='utf-8')
compare_df.to_csv(OUT_DIR / 'compare.csv', encoding='utf-8-sig')
OUT_DIR

## 9. 다음 노트북에서 할 것
- 02: 기본 피처 + 우리 엣지 피처 17개 → 03: + 노드 피처 19개 → 04: + 시간 창 fan / 계좌 통계 (GFP 식) → 05: + 사이클·scatter-gather
- 같은 분할·같은 검증 표본·같은 학습 예산(`ITER_PER_EPOCH × BATCH_SIZE × best_epoch`)으로 돌려서 `compare.csv` 를 이어 붙인다